In [0]:
from datetime import datetime
from pyspark.sql.functions import  lit
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
%run /Workspace/Users/eyskanupuru@etihadppe.ae/azure_edp2_databricks/Production/eag/ey/AP_ANALYTICS/ZFIN_AP_CASHFLOW/schema_repository

In [0]:
#ACDOCA
csv_path = "/mnt/stppeedp/ppeedp/landing/raw/finance/ap/CUSTOM_REPORT/3PIPES_ACDOCA_BUDAT_CUSTOM_REPORT_*.txt"
df=spark.read.option("header", "false").option("delimiter", "|||").schema(custom_report_acdoca_schema).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('acdoca_custom')


In [0]:
#LFA1
table_name = 'LFA1'
base_source_path = "/mnt/stppeedp/ppeedp/landing/raw/finance/ap/REFERENCE_TABLES/LFA1/"
csv_path = base_source_path + '/*.txt'
df=spark.read.option("header", "false").schema(lfa1_schema).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('lfa1')
spark.sql("""select distinct LIFNR, NAME1 from lfa1 where NAME1 is not null """).createOrReplaceTempView('lfa2')

In [0]:
spark.read.parquet('dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/vendor_category_20241127/').createOrReplaceTempView("vendor_category0")
df1 = spark.sql(""" select distinct * from acdoca_custom where (AUGBL is null or AUGBL = '') """)
df1.createOrReplaceTempView('acdoca_custom2')

df2 = spark.sql("""
select 
a.ZUONR AS Assignment,
a.RACCT AS Account,
a.BELNR AS Document_number,
a.BUDAT AS Posting_date,
a.BLDAT AS Document_date,
a.BLART AS Document_type,
a.RBUKRS AS Company_code,
a.GJAHR AS Fiscal_year,
a.AUGBL AS Clearing_document,
a.EBELN AS Purchasing_Document,
a.HSL AS Amount_in_local_currency,
a.OSL AS Amount_in_USD,
--a.LIFNR AS vendor_code,
CASE WHEN a.LIFNR RLIKE '^0+[0-9]+$' THEN REGEXP_REPLACE(a.LIFNR, '^0+', '') ELSE a.LIFNR END AS vendor_code
--lf.NAME1 AS vendor_name
from 
acdoca_custom2 a 
""")
df2.createOrReplaceTempView('acdoca_custom3')

df3 = spark.sql("""
select 
a.Assignment,
a.Account,
a.Document_number,
a.Posting_date,
a.Document_date,
a.Document_type,
a.Company_code,
a.Fiscal_year,
a.Clearing_document,
a.Purchasing_Document,
a.Amount_in_local_currency,
a.Amount_in_USD,
--a.LIFNR AS vendor_code,
a.vendor_code,
coalesce(lf.NAME1, v.Name) AS vendor_name
from 
acdoca_custom3 a 
left join lfa2 lf on a.vendor_code = lf.LIFNR
left join vendor_category0 v on a.vendor_code = v.Vendor
""")
df3 = df3.dropDuplicates()

In [0]:
display(df3.count())

In [0]:
Prod_Final_Path="/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/acdoca_null_clearing_dates_20250701/"
df3.write.mode("overwrite").parquet(Prod_Final_Path)